# DSRL Fine-Tuning

The goal of this notebook is to use Diffusion Steering via Reinforcement Learning (DSRL) to fine-tune a bevore obtained Diffusion Policy. 

This presents an alternative approach for improving a base DP's performance, compared to DynaGuide.

For a thorough comaprison to DynaGuide, DSRL's performance is evaluated using the same scenarios. </br></br>

The following guide closely follows the descrpiton provided by SMLR_G2 2025 and is adjusted to work out of the box with our 2026 DynaGuide repository and respective simulation environments.</br>
The DSRL code is based on the repo by ajwagen: https://github.com/ajwagen/dsrl</br>

We have copied the code physically and have *not* included the DSRL repo as a submodule *on purpose*. This is especially helpful to avoid any version conflicts as it is very likely that submodules get updated indipendently but usually are not backwards compatible out of the box. Therefore to ensure a stabil and working reposiroty, external modules are copied rather than added dynamically.

## 1. Prepare a Diffusion Policy Checkpoint 

As alwyas, first activate your conda environment!

First, take a already trained DIffusiopn Policy (`.pth`). Instead of training your own policy you can also download from the shared storage (see DiffusionPolicy.ipynb).

Now prepare your obtained DP checkpoint for the fine-tuning by running the following command:</br>

```bash
python dsrl_scripts/prepare_checkpoint_for_dsrl.py --dataset dataset/YOUR_CHECKPOINT.pth
```

The prepared dataset will have the same name with `_dsrl` appened to it. Continue with this new dataset!

## 2. Train DSRL

The next step is to create a noise actor which DSRL can later feed to the DP to improve it's performance.</br>
Thos noise vector is created using the bevore obtained DP checkpoint.

Run the following commands:

In [1]:
!ulimit -n 65536

In [11]:
!python dsrl_scripts/dsrl_train_plot.py \
    --agent dataset/ep_300_square_d0_dsrl.pth \
    --out_dir ./dsrl_outputs \
    --env NutAssemblySquare \
    --num_episodes 400 \
    --max_steps 400 \
    --amp \
    --auto_alpha

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_gripper_qpos', 'robot0_eef_pos', 'robot0_eef_quat']
using obs modality: rgb with keys: ['agentview_image', 'robot0_eye_in_hand_image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []
/home/pauld/miniconda3/envs/dynaguide_og/lib/python3.10/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/pauld/miniconda3/envs/dynaguide_og/lib/python

The result can be found in `/dsrl_outputs`.

## 3. Evaluate DSRL

To evaluate the resulting noise vectore use:

In [ ]:
!python dsrl_scripts/dsrl_test.py \
    --dp_ckpt YOUR_DP_CHECKPOINT.pth \
    --noise_actor_ckpt ./dsrl_outputs/TRAINED_NOISE_VECTOR.pth \
    --n_episodes 30 \
    --max_steps 200 \
    --video_path ./dsrl_eval_video

Ommit `--video_path` if you do not want to render a video to the rollouts.</br>
The output video is saved as `dsrl_eval_video/video_{success_rate}.mp4`

**Remark**


If you want to easily compare several differnt noise vectorsd to pick the best, the script `experiemnt_automation/compare_dsrl_noise_vector` can help you. Simply specify the folder containing all noise vectors you want to compare.</br>
Eg.:
```bash
python experiment_automation/compare_dsrl_noise_vectors.py \
    --dp_ckpt dataset/YOUR_DP_CHECKPOINT.pth \
    --ckpt_dir dataset/TRAINED_NOISE_VECTOR \
    --num_workers 6 \
    --n_episodes 100 \
    --max_steps 200 \
    --n_runs_per_policy 5 \
    --out_file /dsrl_comparison/
```

The script will paralelize the task and save the results in a single, structured file.

All experiments conducted using DSRL can be found in Experiments.ipynb